# 02 Traffic Signature Analysis

This notebook explains the same analysis logic that exists in the Python scripts, but in a step-by-step and beginner-friendly notebook format.

Goals:
- load exported traffic signature CSV files
- inspect columns and packet examples
- calculate TCP / UDP / DNS / TLS / QUIC ratios
- compare YouTube, Browsing, and Download traffic signatures

## 1. Imports and Paths

We use `pandas` for table analysis and `pathlib` for file paths.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
SIGNATURE_FILES = {
    "youtube": Path("../traffic_signature_analysis/youtube_signature.csv"),
    "browsing": Path("../traffic_signature_analysis/browsing_signature.csv"),
    "download": Path("../traffic_signature_analysis/download_signature.csv"),
}

SIGNATURE_FILES

{'youtube': WindowsPath('../traffic_signature_analysis/youtube_signature.csv'),
 'browsing': WindowsPath('../traffic_signature_analysis/browsing_signature.csv'),
 'download': WindowsPath('../traffic_signature_analysis/download_signature.csv')}

## 2. Load the CSV Files

Each CSV already comes from a traffic-specific tshark filter.
We read them into pandas DataFrames.

In [3]:
def load_signature_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, keep_default_na=False, low_memory=False)
    df.columns = [column.strip() for column in df.columns]
    return df.fillna("")


dataframes = {label: load_signature_csv(path) for label, path in SIGNATURE_FILES.items()}

for label, df in dataframes.items():
    print(label, df.shape)

youtube (68479, 13)
browsing (35121, 13)
download (916205, 13)


In [4]:
dataframes["youtube"].head()

,frame.time_relative,ip.src,ip.dst,ip.proto,tcp.srcport,tcp.dstport,udp.srcport,udp.dstport,frame.len,dns.qry.name,tls.handshake.extensions_server_name,_ws.col.protocol,label
0,0.669058,10.100.1.194,10.10.101.50,17,,,55436,53,78,auth.grammarly.com,,DNS,youtube
1,0.688773,10.10.101.50,10.100.1.194,17,,,53,55436,206,auth.grammarly.com,,DNS,youtube
2,0.765874,10.100.1.194,10.10.101.50,17,,,55617,53,88,applet-bundles.grammarly.net,,DNS,youtube
3,0.772622,10.10.101.50,10.100.1.194,17,,,53,55617,152,applet-bundles.grammarly.net,,DNS,youtube
4,1.238514,192.178.202.94,10.100.1.194,17,,,443,63665,318,,,QUIC,youtube


## 3. Important Columns

The most important columns for this analysis are:

- `ip.proto`: protocol number from the packet data itself
- `_ws.col.protocol`: protocol text written by Wireshark/tshark
- `dns.qry.name`: DNS query name if the packet is DNS
- `tcp.srcport` / `tcp.dstport`: TCP ports
- `udp.srcport` / `udp.dstport`: UDP ports
- `frame.len`: packet size in bytes

Important note:
- `ip.proto = 6` means TCP
- `ip.proto = 17` means UDP

Those meanings are standardized. The values come from your packet data.

## 4. Helper Functions

These helper functions recreate the same logic used in `src/analyze_traffic_signatures.py`.

In [5]:
def numeric_series(dataframe: pd.DataFrame, column_name: str) -> pd.Series:
    return pd.to_numeric(dataframe[column_name], errors="coerce").fillna(0)


def protocol_mask(dataframe: pd.DataFrame, protocol_name: str) -> pd.Series:
    protocol_text = dataframe["_ws.col.protocol"].astype(str).str.upper()
    ip_src = dataframe["ip.src"].astype(str).str.strip()
    ip_dst = dataframe["ip.dst"].astype(str).str.strip()
    ip_proto = numeric_series(dataframe, "ip.proto")
    tcp_src = dataframe["tcp.srcport"].astype(str).str.strip()
    tcp_dst = dataframe["tcp.dstport"].astype(str).str.strip()
    udp_src = dataframe["udp.srcport"].astype(str).str.strip()
    udp_dst = dataframe["udp.dstport"].astype(str).str.strip()
    dns_name = dataframe["dns.qry.name"].astype(str).str.strip()
    tls_name = dataframe["tls.handshake.extensions_server_name"].astype(str).str.strip()

    if protocol_name == "tcp":
        return (ip_proto == 6) | (tcp_src != "") | (tcp_dst != "")

    if protocol_name == "udp":
        return (ip_proto == 17) | (udp_src != "") | (udp_dst != "")

    if protocol_name == "dns":
        return (dns_name != "") | protocol_text.str.contains("DNS", na=False)

    if protocol_name == "quic":
        return protocol_text.str.contains("QUIC", na=False)

    if protocol_name == "tls":
        tcp_443 = (tcp_src == "443") | (tcp_dst == "443")
        explicit_tls = (tls_name != "") | protocol_text.str.contains("TLS", na=False) | tcp_443

        flow_keys = (
            ip_src.where(ip_src <= ip_dst, ip_dst)
            + "|"
            + ip_dst.where(ip_src <= ip_dst, ip_src)
            + "|"
            + tcp_src.where(tcp_src <= tcp_dst, tcp_dst)
            + "|"
            + tcp_dst.where(tcp_src <= tcp_dst, tcp_src)
        )

        tcp_flow_mask = (ip_proto == 6) & ((tcp_src != "") | (tcp_dst != ""))
        tls_flow_keys = set(flow_keys[tcp_flow_mask & explicit_tls])
        flow_based_tls = tcp_flow_mask & flow_keys.isin(tls_flow_keys)
        return explicit_tls | flow_based_tls

    raise ValueError(f"Unsupported protocol name: {protocol_name}")


def protocol_stats(dataframe: pd.DataFrame, protocol_name: str) -> dict:
    mask = protocol_mask(dataframe, protocol_name)
    total_packets = len(dataframe)
    total_bytes = numeric_series(dataframe, "frame.len").sum()
    packet_count = int(mask.sum())
    byte_count = int(numeric_series(dataframe.loc[mask], "frame.len").sum())

    return {
        "packet_count": packet_count,
        "packet_ratio": (packet_count / total_packets * 100) if total_packets else 0,
        "byte_count": byte_count,
        "byte_ratio": (byte_count / total_bytes * 100) if total_bytes else 0,
    }

## 5. Why `ip.proto == 6` and `ip.proto == 17`?

This is the core logic:

- `6` is the standard protocol number for TCP
- `17` is the standard protocol number for UDP

The number comes from your packet data.
The meaning of the number comes from the IP protocol standard.

In [6]:
dataframes["youtube"][['ip.proto', '_ws.col.protocol', 'tcp.srcport', 'udp.srcport']].head(10)

,ip.proto,_ws.col.protocol,tcp.srcport,udp.srcport
0,17,DNS,,55436
1,17,DNS,,53
2,17,DNS,,55617
3,17,DNS,,53
4,17,QUIC,,443
5,17,QUIC,,443
6,17,DNS,,60344
7,17,DNS,,65070
8,17,DNS,,49664
9,17,DNS,,53


## 6. Example: Calculate QUIC Ratio for YouTube

This shows how one of the main findings is calculated.

In [7]:
youtube_df = dataframes["youtube"]
quic_mask = protocol_mask(youtube_df, "quic")

quic_packet_count = int(quic_mask.sum())
youtube_total_packets = len(youtube_df)
quic_ratio = quic_packet_count / youtube_total_packets * 100

print("QUIC packets:", quic_packet_count)
print("Total packets:", youtube_total_packets)
print("QUIC ratio (%):", round(quic_ratio, 2))

QUIC packets: 68132
Total packets: 68479
QUIC ratio (%): 99.49


## 7. Calculate All Protocol Ratios

We now repeat the same logic for all traffic types.

In [8]:
def summarize_dataframe(label: str, dataframe: pd.DataFrame) -> dict:
    tcp = protocol_stats(dataframe, "tcp")
    udp = protocol_stats(dataframe, "udp")
    dns = protocol_stats(dataframe, "dns")
    tls = protocol_stats(dataframe, "tls")
    quic = protocol_stats(dataframe, "quic")

    return {
        "label": label,
        "total_packets": len(dataframe),
        "tcp_packet_ratio": round(tcp["packet_ratio"], 2),
        "udp_packet_ratio": round(udp["packet_ratio"], 2),
        "dns_packet_ratio": round(dns["packet_ratio"], 2),
        "tls_byte_ratio": round(tls["byte_ratio"], 2),
        "quic_packet_ratio": round(quic["packet_ratio"], 2),
        "dns_packet_count": int(dns["packet_count"]),
        "unique_destination_ips": int(dataframe['ip.dst'].astype(str).str.strip().replace('', pd.NA).dropna().nunique()),
        "unique_dns_queries": int(dataframe['dns.qry.name'].astype(str).str.strip().replace('', pd.NA).dropna().nunique()),
    }


summary_rows = [summarize_dataframe(label, df) for label, df in dataframes.items()]
summary_df = pd.DataFrame(summary_rows)
summary_df

,label,total_packets,tcp_packet_ratio,udp_packet_ratio,dns_packet_ratio,tls_byte_ratio,quic_packet_ratio,dns_packet_count,unique_destination_ips,unique_dns_queries
0,youtube,68479,0.00,100.00,0.34,0.05,99.49,234,18,42
1,browsing,35121,98.74,1.26,2.09,98.40,0.83,735,131,106
2,download,916205,99.98,0.02,0.00,99.99,0.02,0,100,0


## 8. Read the Results Like a Networking Student

Typical interpretation:

- very high QUIC + very high UDP -> streaming-like traffic
- very high TCP + very high TLS -> download-like encrypted traffic
- high TCP + visible DNS + many destinations -> browsing-like traffic

In [9]:
summary_df[['label', 'tcp_packet_ratio', 'udp_packet_ratio', 'quic_packet_ratio', 'dns_packet_ratio', 'tls_byte_ratio', 'unique_destination_ips']]

,label,tcp_packet_ratio,udp_packet_ratio,quic_packet_ratio,dns_packet_ratio,tls_byte_ratio,unique_destination_ips
0,youtube,0.00,100.00,99.49,0.34,0.05,18
1,browsing,98.74,1.26,0.83,2.09,98.40,131
2,download,99.98,0.02,0.02,0.00,99.99,100


## 9. Top Protocol Labels in Each CSV

This is similar to looking at Wireshark protocol labels, but from the exported CSV.

In [10]:
for label, df in dataframes.items():
    print(f"\n--- {label.upper()} ---")
    print(df['_ws.col.protocol'].astype(str).str.strip().replace('', 'UNKNOWN').value_counts().head(10))


--- YOUTUBE ---
_ws.col.protocol
QUIC    68132
DNS       234
UDP       113
Name: count, dtype: int64

--- BROWSING ---
_ws.col.protocol
TCP        24629
TLSv1.3     7968
TLSv1.2     1324
DNS          735
QUIC         290
SSL          172
HTTP           2
TLSv1          1
Name: count, dtype: int64

--- DOWNLOAD ---
_ws.col.protocol
TCP         761218
TLSv1.3     153341
SSLv2          768
TLSv1.2        633
QUIC           146
SSL             80
HTTP            16
HTTP/XML         2
TLSv1            1
Name: count, dtype: int64


## 10. Optional: Re-run the Script Versions

If you want, you can re-run the `.py` files directly from the notebook.
Leave these cells commented out unless you want to regenerate files.

In [11]:
# import subprocess
# subprocess.run(['python', '../src/traffic_signature_analysis.py'], check=True)
# subprocess.run(['python', '../src/analyze_traffic_signatures.py'], check=True)

## 11. Final Learning Notes

By the end of this notebook, you should understand:

- how tshark-exported packet data becomes a pandas DataFrame
- how protocol ratios can be calculated with boolean masks
- why `ip.proto = 6` means TCP and `ip.proto = 17` means UDP
- how YouTube, Browsing, and Download traffic can be distinguished with protocol patterns